# TEKNOFEST KPI — Colab (sıfırdan kurulum)

**Kural:** kod GitHub’dan, veri Drive’dan. Zip yok.

| Ne | Nerede |
|---|---|
| Kod / scriptler | `git clone` → `/content/teknofest-video-ajan` |
| Videolar + gold | Drive `MyDrive/KAIZEN_KPI/data/` |
| KPI sonuçları | Drive `.../data/exports/` + `predictions_wide/` |

## Drive’da bir kez (PC’den)

```
MyDrive/KAIZEN_KPI/
  data/
    videos/
      accident/*.mp4
      near_miss/*.mp4
      normal/*.mp4
    exports/
      gold_labels_hepsi.json
```

`video: 0` görüyorsan neredeyse her zaman videolar yanlış klasörde (ör. `KAIZEN_KPI/` kökünde veya zip içinde). Yukarıdaki yol şart.

## Colab

1. Runtime → Change runtime type → **T4 GPU**
2. Bu defteri aç (GitHub’dan veya File → Upload)
3. Hücreleri **sırayla** çalıştır

Kod güncellendiğinde: sadece **“Repoyu çek”** hücresini tekrar çalıştır (`git pull`). Zip / Drive kod güncellemesi yok.

### 1) GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime → Change runtime type → T4 GPU seç.'

### 2) Drive bağla (sadece veri)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/KAIZEN_KPI')
for sub in ('data/videos/accident', 'data/videos/near_miss', 'data/videos/normal',
            'data/exports', 'data/predictions_wide'):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

print('Drive kökü:', DRIVE_ROOT)
print('Beklenen videolar:', DRIVE_ROOT / 'data/videos')

### 3) Repoyu GitHub’dan çek

`main` = güncel Zehra + ekip kodu. Her kod değişikliğinde bu hücreyi yeniden çalıştırman yeterli.

In [ ]:
import os

REPO = '/content/teknofest-video-ajan'
REPO_URL = 'https://github.com/TulinBabalikKopmaz/KAIZEN_Teknofest26_DilAjanlar-_VideoAnalizKararSistemi.git'
BRANCH = 'main'

if not os.path.exists(REPO):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO}
else:
    %cd {REPO}
    !git fetch --depth 1 origin {BRANCH}
    !git checkout -B {BRANCH} origin/{BRANCH}
    !git reset --hard origin/{BRANCH}

%cd {REPO}
!git log -1 --oneline
!ls scripts/run_kpi_wide.py scripts/colab_kpi_bootstrap.py prompts/video_label_prompt.txt

### 4a) Pip + Drive bağla (Ollama yok)

Bu hücre hızlı biter. Videoların bağlandığını burada görürsün.

In [ ]:
%cd /content/teknofest-video-ajan
!python -u scripts/colab_kpi_bootstrap.py \
  --drive-root /content/drive/MyDrive/KAIZEN_KPI \
  --repo /content/teknofest-video-ajan \
  --model qwen2.5vl:7b \
  --skip-ollama

### 4b) Ollama + Qwen model indir

İndirme **5–15 dk** sürebilir; progress satırları akmalı.  
Biterse `model OK: qwen2.5vl:7b` görürsün. Görmezsen hücreyi durdurup tekrar çalıştır.

In [ ]:
%cd /content/teknofest-video-ajan
# Önce GitHub'daki güncel bootstrap'u al (bu oturumda eski kod kaldıysa)
!git fetch --depth 1 origin main
!git reset --hard origin/main

!python -u scripts/colab_kpi_bootstrap.py --only-ollama --model qwen2.5vl:7b
!ollama list
print('Ollama bitti — sonraki teşhis / KPI hücrelerine geç.')

### 5) Veri teşhisi (`video: 0` buradan anlaşılır)

Gold Drive’da olabilir, videolar başka yerde olabilir. Bu hücre her iki yolu da tarar.

In [ ]:
from pathlib import Path

VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
drive_videos = Path('/content/drive/MyDrive/KAIZEN_KPI/data/videos')
repo_videos = Path('/content/teknofest-video-ajan/data/videos')
gold = Path('/content/teknofest-video-ajan/data/exports/gold_labels_hepsi.json')

def list_vids(root: Path):
    if not root.exists():
        return []
    return [p for p in root.rglob('*') if p.suffix.lower() in VIDEO_EXTS]

dv = list_vids(drive_videos)
rv = list_vids(repo_videos)

print('Drive videos path :', drive_videos, 'exists=', drive_videos.exists())
print('Repo  videos path :', repo_videos, 'symlink=', repo_videos.is_symlink())
if repo_videos.is_symlink():
    print('  → points to      :', repo_videos.resolve())
print('Drive video count :', len(dv))
print('Repo  video count :', len(rv))
print('gold              :', gold.exists(), gold)

print('\nDrive/videos alt klasörler:')
if drive_videos.exists():
    for p in sorted(drive_videos.iterdir()):
        n = len(list_vids(p)) if p.is_dir() else 0
        print(f'  {p.name}/  ({n} video)' if p.is_dir() else f'  {p.name}')
else:
    print('  (klasör yok)')

if dv[:5]:
    print('\nÖrnek dosyalar:')
    for p in dv[:5]:
        print(' ', p.relative_to(drive_videos))

# Sık hata: videolar Drive kökünde veya yanlış yerde
wrong_roots = [
    Path('/content/drive/MyDrive/KAIZEN_KPI'),
    Path('/content/drive/MyDrive/KAIZEN_KPI/videos'),
    Path('/content/drive/MyDrive/videos'),
]
print('\nYanlış yerde video var mı?')
for wr in wrong_roots:
    found = [p for p in list_vids(wr) if 'data/videos' not in str(p).replace('\\', '/')]
    # sadece doğrudan bu kökte / videos altında, data/videos dışında
    direct = []
    if wr.exists():
        for p in wr.rglob('*'):
            if p.suffix.lower() not in VIDEO_EXTS:
                continue
            s = str(p)
            if '/data/videos/' in s.replace('\\', '/'):
                continue
            direct.append(p)
    print(f'  {wr}: {len(direct)}')
    for p in direct[:3]:
        print('   ', p)

assert gold.exists(), (
    'gold_labels_hepsi.json yok. PC\'den şuraya koy:\n'
    '  MyDrive/KAIZEN_KPI/data/exports/gold_labels_hepsi.json'
)
assert len(rv) >= 18, (
    f'Repo tarafında {len(rv)} video var (en az 18 lazım).\n'
    'PC\'den mp4\'leri şuraya yükle:\n'
    '  MyDrive/KAIZEN_KPI/data/videos/accident|near_miss|normal/\n'
    'Sonra bu hücreyi tekrar çalıştır (bootstrap\'u da yeniden koşabilirsin).'
)

### 6) KPI çalıştır (18 video)

Sonuçlar Drive’a yazılır (~30–90 dk, T4).

In [ ]:
%cd /content/teknofest-video-ajan
!python -u scripts/run_kpi_wide.py \
  --n 18 \
  --seed 42 \
  --model qwen2.5vl:7b \
  --pred-dir data/predictions_wide \
  --no-second-look

print('--- özet ---')
!ls -la data/exports/kpi_wide*.csv 2>/dev/null || true
!ls data/exports/kpi_wide* 2>/dev/null || ls data/exports/

### Kod güncellemesi (sonraki seferler)

```text
1) PC'de commit + push → GitHub main
2) Colab'de sadece hücre 3 (git pull / reset) → tekrar çalıştır
3) Gerekirse hücre 6 (KPI)
```

Drive’a zip atmana gerek yok. Videolar / gold Drive’da kalır.